# Model Selection and Clustering

This notebook encapsulates the complete pipeline for selecting the large language models (LLMs) to be subjected to study.

The process consists of:
1. **Data Collection**: Fetching model details from predefined Hugging Face collections.
2. **Filtering**: Applying a set of rules to ignore specialized, quantized, reasoning/thinking, and MoE models.
3. **Prioritization**: Choosing instruction-tuned models over their base versions.
4. **Clustering**: Grouping models into sizes (Tiny, Small, Standard, Mid-range) using Jenks Natural Breaks.
5. **Selection**: Selecting the most recent or largest parameter model available within each category for each family.

### Setup & Imports
We import necessary libraries to handle API requests (`huggingface_hub`), data manipulation (`pandas`), and clustering (`jenkspy`).

In [100]:
import pandas as pd
import jenkspy
import re
import os
import json
from huggingface_hub import HfApi, get_collection, hf_hub_download
from transformers import AutoTokenizer, AutoConfig

### 1. Configuration and Filtering Rules
Here we define our constants and parameters for model collection. We set maximum limits, regex patterns to extract parameter sizes, and lists of keywords/types to ignore (e.g., GGUF, MoE, Vision models) ensuring we only capture standard text generation base or instruct models.

In [101]:
# Maximum parameter count in billions (hardware constraints)
MAX_PARAMS = 50.0

# Path to cache manual entries
MANUAL_CACHE_FILE = "manual_model_data.json"

def load_manual_cache():
    if os.path.exists(MANUAL_CACHE_FILE):
        with open(MANUAL_CACHE_FILE, "r") as f:
            return json.load(f)
    return {}

def save_manual_cache(cache):
    with open(MANUAL_CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=4)

# Load existing cache
MANUAL_CACHE = load_manual_cache()

# Regex Patterns
PARAM_PATTERN = re.compile(r'(\d+(?:\.\d+)?)\s*[Bb]', re.IGNORECASE)
M_PARAM_PATTERN = re.compile(r'(\d+(?:\.\d+)?)\s*m', re.IGNORECASE)

# Matches patterns like 30B-A3B (MoE architecture)
MOE_A_PATTERN = re.compile(r'(\d+(?:\.\d+)?)B-A(\d+(?:\.\d+)?)B', re.IGNORECASE)
# Matches patterns like 17B-128E (LLaMA 4 MoE architecture)
MOE_A_LLAMA_PATTERN = re.compile(r'(\d+(?:\.\d+)?)B-(\d+(?:\.\d+)?)E', re.IGNORECASE)
# Specifically for models that only list activated parameters (e.g. -A2.7B)
MOE_A_ONLY_PATTERN = re.compile(r'-A(\d+(?:\.\d+)?)[B|E]', re.IGNORECASE)
# Specifically for Gemma 4 Per-Layer Embeddings (e.g. -E4B)
GEMMA_E_PATTERN = re.compile(r'-E(\d+(?:\.\d+)?)[B|b]', re.IGNORECASE)
GENERIC_MOE_PATTERN = re.compile(r'moe', re.IGNORECASE)

# Instruction Suffixes
PRIORITIZE_SUFFIXES = ["-it", "-instruct", "-chat", "-instruction", "-hf"]

# Types to ignore
IGNORE_TYPES = {
    "audio-to-audio",
    "text-to-audio",
    "audio-text-to-audio",
    "text-to-image",
    "image-to-image",
    "image-text-to-image"
}

# Specific words to ignore in model names (case-insensitive)
IGNORE_WORDS = {
    # Quantized versions
    "gguf", "awq", "gptq", "quant", "quantized", "int4", "int8", "fp8", "4bit", "6bit", "8bit",
    # Specialized versions
    "special", "speciale", "vision", "image", "audio", "code", "coder", "math", "ocr", "guard",
    "pytorch", "keras", "flax", "cpp", "tflite", "mlx", "terminus", "exp", "shield",
    # Not instruction-tuned versions
    "base", "-pt",
    # Others
    "deprecated"
}

# Specific models to ignore by exact model ID
IGNORE_MODELS = {
    # Hugging Face base models that are not instruction-tuned, confusing the prioritization logic.
    "meta-llama/Llama-2-7b-hf",
    "meta-llama/Llama-2-13b-hf",
    # Gemma 4 31B ignored due to "repetition collapse"
    # bug that is not fixed as of April 2026.
    "google/gemma-4-31B",
    "google/gemma-4-31B-it"
}

# Specific models to explicitly identify as MoE
MOE_MODELS = {
    "deepseek-ai/DeepSeek-V2",
    "deepseek-ai/DeepSeek-V2-Chat", 
    "deepseek-ai/DeepSeek-V2-Lite-Chat",
    "deepseek-ai/DeepSeek-V2.5",
    "deepseek-ai/DeepSeek-V2.5-1210",
    "deepseek-ai/DeepSeek-V3",
    "deepseek-ai/DeepSeek-V3-Base",
    "DeepSeek-V3.1-Base",
    "DeepSeek-V3.1",
    "DeepSeek-V3.2",
}

# Specific models to explicitly identify as Thinking/Reasoning
THINKING_MODELS = {"deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"}

# Specific words to identify as Thinking/Reasoning (case-insensitive)
THINKING_WORDS = {"reason", "reasoning", "think", "thinking"}

### 2. Helper Functions
These functions parse model names to extract parameter sizes and query the Hugging Face API (`get_model_details`). During the API query, the model is checked against the ignoring rules before being accepted.

In [102]:
def format_to_billions(num):
    """Converts raw integer or float to 'X.XX' string."""
    if num is None or num == "N/A":
        return "N/A"
    try:
        return f"{float(num) / 1e9:.2f}"
    except (ValueError, TypeError):
        return str(num)

def prompt_for_missing_value(model_id, field_name):
    """Prompts the user to manually input a missing value for a model and caches it."""
    # Check cache first
    cache_key = f"{model_id}|{field_name}"
    if cache_key in MANUAL_CACHE:
        return MANUAL_CACHE[cache_key]
        
    while True:
        user_input = input(f"Enter {field_name} (in billions, e.g., 7.0) for {model_id}: ").strip()
        if not user_input:
            print("Value is required. Please enter a number.")
            continue
        try:
            val = f"{float(user_input):.2f}"
            # Update cache and save
            MANUAL_CACHE[cache_key] = val
            save_manual_cache(MANUAL_CACHE)
            return val
        except ValueError:
            print("Invalid input. Please enter a numeric value.")

def extract_params_from_name(model_id):
    """
    Tries to find parameter count and density details in the model name.
    Returns (discovered_total, discovered_active, is_moe)
    """
    model_name_only = model_id.split('/')[-1]
    
    # 1. Check for explicit MoE Total and Active (e.g., 14B-A2.7B)
    moe_a_match = MOE_A_PATTERN.search(model_name_only)
    if moe_a_match:
        total = f"{float(moe_a_match.group(1)):.2f}"
        active = f"{float(moe_a_match.group(2)):.2f}"
        return total, active, True

    # 1b. Check for LLaMA 4 MoE pattern (e.g., 17B-128E)
    moe_a_llama_match = MOE_A_LLAMA_PATTERN.search(model_name_only)
    if moe_a_llama_match:
        active = f"{float(moe_a_llama_match.group(1)):.2f}"
        return "N/A", active, True

    # 2. Check for Active only pattern (e.g., -A2.7B)
    moe_a_only_match = MOE_A_ONLY_PATTERN.search(model_name_only)
    if moe_a_only_match:
        active = f"{float(moe_a_only_match.group(1)):.2f}"
        return "N/A", active, True
    
    # 3. Check for Gemma 4 'E' pattern (e.g., -E4B)
    gemma_e_match = GEMMA_E_PATTERN.search(model_name_only)
    if gemma_e_match:
        active = f"{float(gemma_e_match.group(1)):.2f}"
        # For Gemma 'E' models, total is often not specified in name
        return "N/A", active, True

    # 4. Fallback to generic parameter extraction
    is_moe = bool(GENERIC_MOE_PATTERN.search(model_name_only))
    
    match = PARAM_PATTERN.search(model_name_only)
    if match:
        try:
            val = f"{float(match.group(1)):.2f}"
            return val, val, is_moe
        except ValueError:
            pass
    else:
        alt_match = M_PARAM_PATTERN.search(model_name_only)
        if alt_match:
            try:
                val = f"{(float(alt_match.group(1))/1000):.2f}"
                return val, val, is_moe
            except ValueError:
                pass
                
    return "N/A", "N/A", is_moe

def check_is_thinking(model_id):
    """
    Checks if a model is a reasoning/thinking model based on:
    0. Hardcoded NON_THINKING_MODELS list (overrides all)
    1. Hardcoded THINKING_MODELS list
    2. Name containing thinking words
    3. Chat template containing thinking tags
    """
    if model_id in THINKING_MODELS:
        return True
        
    model_id_lower = model_id.lower()
    if any(word in model_id_lower for word in THINKING_WORDS):
        return True
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        # Check for the presence of 'think' tags in the default chat template
        if tokenizer.chat_template:
            template = tokenizer.chat_template
            if isinstance(template, list):
                template = str(template)
            thinking_tags = ["<think>", "<reasoning>", "<|think|>", "<|channel>thought"]
            if any(tag in template for tag in thinking_tags):
                return True
    except Exception:
        pass
        
    return False

def get_max_model_length(model_id):
    """
    Attempts to fetch the maximum natively supported context length from the model's config.
    """
    try:
        config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
        
        # Some multimodal models (like Gemma 3) store their text sequence length inside a text_config
        target_config = config.text_config if hasattr(config, "text_config") else config
        
        # Check standard config attributes used by different architectures for max context length
        if hasattr(target_config, "max_position_embeddings"):
            return target_config.max_position_embeddings
        elif hasattr(target_config, "n_positions"):
            return target_config.n_positions
        elif hasattr(target_config, "seq_length"):
            return target_config.seq_length
        elif hasattr(target_config, "max_sequence_length"):
            return target_config.max_sequence_length
            
    except Exception:
        pass
        
    return "N/A"

def get_total_parameters_from_config(model_id):
    """
    Attempts to calculate the total number of parameters accurately by instantiating an 
    empty model from the config and extracting the parameter count directly.
    """
    try:
        from transformers import AutoModelForCausalLM
        import accelerate
        
        config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
        # Initialize an empty model using accelerate to avoid downloading weights
        with accelerate.init_empty_weights():
            model = AutoModelForCausalLM.from_config(config, trust_remote_code=True)
            
        return format_to_billions(model.num_parameters())
    except Exception:
        return "N/A"

def get_model_details(api, model_id):
    """Fetches details and returns (model_dict, status_string)."""
    try:
        # 1. Filter out by model id if it's in the ignore list
        if model_id in IGNORE_MODELS:
            return None, "Manually ignored"

        # 2. Filter out by ignored words in the name
        model_id_lower = model_id.lower()
        for word in IGNORE_WORDS:
            if word.lower() in model_id_lower:
                return None, f"Ignored word '{word}'"

        # Fetch model info only after local name checks pass
        info = api.model_info(model_id)

        # 3. Filter out ignored model types
        if info.pipeline_tag and info.pipeline_tag.lower() in IGNORE_TYPES:
            return None, f"Ignored type '{info.pipeline_tag}'"

        if info.tags:
            for tag in info.tags:
                if tag.lower() in IGNORE_TYPES:
                    return None, f"Ignored tag '{tag}'"

        # 4. Extract Release Date
        release_date = info.created_at.strftime("%Y-%m-%d") if info.created_at else "N/A"

        # 5. Extract Parameter Counts (Metadata -> Config -> Name Fallback -> Manual Prompt)
        total_params = "N/A"
        activated_params = "N/A"
        is_moe = False
        
        # A. Check safetensors metadata first for total params (Source of Truth)
        if info.safetensors and "total" in info.safetensors:
            total_params = format_to_billions(info.safetensors["total"])
            
        # B. Check config for total params
        if total_params == "N/A":
            # Attempt to use AutoConfig and empty model trick
            total_params = get_total_parameters_from_config(model_id)
            
        # C. Discover counts from name as fallback
        n_total, n_active, n_is_moe = extract_params_from_name(model_id)
        
        # D. Identification logic
        is_moe = n_is_moe
        if info.tags and any("moe" in tag.lower() for tag in info.tags):
            is_moe = True
        if model_id in MOE_MODELS:
            is_moe = True
            
        # E. Parameters Assignment logic
        # If metadata and config are both missing total, use name's total
        if total_params == "N/A":
            total_params = n_total
        
        # If still N/A, prompt manually
        if total_params == "N/A":
            total_params = prompt_for_missing_value(model_id, "Total Parameters")
        
        if is_moe:
            # For MoE, use 'active' from name if found, else fallback to N/A
            activated_params = n_active if n_active != "N/A" else "N/A"
            if activated_params == "N/A":
                # Only prompt if it's definitely MoE and we don't have active params
                activated_params = prompt_for_missing_value(model_id, "Activated Parameters")
        else:
            # For dense, both are the same
            activated_params = total_params

        if total_params != "N/A" and float(total_params) > MAX_PARAMS:
            return None, f"Exceeds max params ({total_params}B > {MAX_PARAMS}B)"
            
        # 6. Determine Thinking Mode
        is_thinking = check_is_thinking(model_id)

        # 7. Extract Max Context Length
        max_model_length = get_max_model_length(model_id)

        return {
            "model": model_id,
            "total_parameters": total_params,
            "activated_parameters": activated_params,
            "max_model_length": max_model_length,
            "is_moe": is_moe,
            "is_thinking": is_thinking,
            "release_date": release_date
        }, "Accepted"
    except Exception as e:
        return None, f"API Error: {e}"

### 3. Fetching Data from Hugging Face
We iterate over the list of chosen collections, fetch the models in each collection, and apply our filtering logic to compile a list of viable models.

In [103]:
COLLECTIONS = [
    # Original Qwen collection is ignored since its models are not natively supported by Hugging Face Transformers,
    # needing to run remote code that fails since it is not compatible with current package versions.
    # The Qwen 1.5 collection contains re-released model weights from the original Qwen collection,
    # but in a format that is natively supported.
    "Qwen/qwen15",
    "Qwen/qwen2",
    "Qwen/qwen25",
    "Qwen/qwen3",
    "Qwen/qwen35",
    "meta-llama/llama-2-family",
    "meta-llama/meta-llama-3",
    "meta-llama/llama-31",
    "meta-llama/llama-32",
    "meta-llama/llama-33",
    "meta-llama/llama-4",
    "deepseek-ai/deepseek-llm",
    "deepseek-ai/deepseek-v2",
    "deepseek-ai/deepseek-v25",
    "deepseek-ai/deepseek-r1",
    "deepseek-ai/deepseek-v3",
    "deepseek-ai/deepseek-v31",
    "deepseek-ai/deepseek-v32",
    "google/gemma-release",
    "google/gemma-2-release",
    "google/gemma-3-release",
    "google/gemma-3n",
    "google/gemma-4",
    "mistralai/ministral-3"
]

api = HfApi()
rows = []

for col_id in COLLECTIONS:
    print(f"\n📦 Collection: {col_id}")
    try:
        collection = get_collection(col_id)
        for item in collection.items:
            if item.item_type == "model":
                details, reason = get_model_details(api, item.item_id)
                
                if details:
                    details["collection"] = col_id
                    details["collection_name"] = collection.title
                    rows.append(details)
                    print(f"  ✓ {item.item_id}")
                else:
                    print(f"  ✗ {item.item_id} (Reason: {reason})")
    except Exception as e:
        print(f"  [!] Collection Error: {e}")



📦 Collection: Qwen/qwen15
  ✗ Qwen/Qwen1.5-110B-Chat (Reason: Exceeds max params (111.21B > 50.0B))
  ✗ Qwen/Qwen1.5-110B (Reason: Exceeds max params (111.21B > 50.0B))
  ✓ Qwen/Qwen1.5-32B
  ✓ Qwen/Qwen1.5-32B-Chat
  ✗ Qwen/Qwen1.5-72B-Chat (Reason: Exceeds max params (72.29B > 50.0B))
  ✗ Qwen/Qwen1.5-72B (Reason: Exceeds max params (72.29B > 50.0B))
  ✓ Qwen/Qwen1.5-MoE-A2.7B
  ✓ Qwen/Qwen1.5-MoE-A2.7B-Chat
  ✓ Qwen/Qwen1.5-4B
  ✓ Qwen/Qwen1.5-4B-Chat
  ✗ Qwen/Qwen1.5-32B-Chat-GPTQ-Int4 (Reason: Ignored word 'int4')
  ✗ Qwen/Qwen1.5-32B-Chat-GGUF (Reason: Ignored word 'gguf')
  ✗ Qwen/Qwen1.5-32B-Chat-AWQ (Reason: Ignored word 'awq')
  ✓ Qwen/Qwen1.5-0.5B
  ✓ Qwen/Qwen1.5-0.5B-Chat
  ✓ Qwen/Qwen1.5-1.8B
  ✓ Qwen/Qwen1.5-7B
  ✓ Qwen/Qwen1.5-14B
  ✓ Qwen/Qwen1.5-14B-Chat
  ✗ Qwen/Qwen1.5-0.5B-Chat-GPTQ-Int4 (Reason: Ignored word 'int4')
  ✗ Qwen/Qwen1.5-7B-Chat-GPTQ-Int4 (Reason: Ignored word 'int4')
  ✓ Qwen/Qwen1.5-1.8B-Chat
  ✓ Qwen/Qwen1.5-7B-Chat
  ✗ Qwen/Qwen1.5-0.5B-Chat-GPTQ-

[transformers] `rope_parameters`'s high_freq_factor field must be greater than low_freq_factor, got high_freq_factor=1.0 and low_freq_factor=1.0


  ✗ meta-llama/Llama-4-Scout-17B-16E-Instruct (Reason: Exceeds max params (109.00B > 50.0B))


[transformers] `rope_parameters`'s high_freq_factor field must be greater than low_freq_factor, got high_freq_factor=1.0 and low_freq_factor=1.0


  ✗ meta-llama/Llama-4-Scout-17B-16E (Reason: Exceeds max params (109.00B > 50.0B))
  ✗ meta-llama/Llama-4-Maverick-17B-128E-Instruct (Reason: Exceeds max params (401.58B > 50.0B))
  ✗ meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8 (Reason: Ignored word 'fp8')
  ✗ meta-llama/Llama-4-Maverick-17B-128E (Reason: Exceeds max params (401.58B > 50.0B))
  ✗ meta-llama/Llama-4-Scout-17B-16E-Instruct-Original (Reason: Exceeds max params (109.00B > 50.0B))
  ✗ meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8-Original (Reason: Ignored word 'fp8')
  ✗ meta-llama/Llama-4-Scout-17B-16E-Original (Reason: Exceeds max params (109.00B > 50.0B))
  ✗ meta-llama/Llama-4-Maverick-17B-128E-Instruct-Original (Reason: Exceeds max params (400.00B > 50.0B))
  ✗ meta-llama/Llama-4-Maverick-17B-128E-Original (Reason: Exceeds max params (400.00B > 50.0B))
  ✗ meta-llama/Llama-Prompt-Guard-2-22M (Reason: Ignored word 'guard')
  ✗ meta-llama/Llama-Prompt-Guard-2-86M (Reason: Ignored word 'guard')
  ✗ meta-llama/L

...

  ✓ mistralai/Ministral-3-14B-Instruct-2512


[transformers] The tokenizer you are loading from 'mistralai/Ministral-3-8B-Instruct-2512' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


  ✓ mistralai/Ministral-3-8B-Instruct-2512


[transformers] The tokenizer you are loading from 'mistralai/Ministral-3-3B-Instruct-2512' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


  ✓ mistralai/Ministral-3-3B-Instruct-2512
  ✓ mistralai/Ministral-3-14B-Reasoning-2512
  ✓ mistralai/Ministral-3-8B-Reasoning-2512
  ✓ mistralai/Ministral-3-3B-Reasoning-2512
  ✗ mistralai/Ministral-3-14B-Base-2512 (Reason: Ignored word 'base')
  ✗ mistralai/Ministral-3-8B-Base-2512 (Reason: Ignored word 'base')
  ✗ mistralai/Ministral-3-3B-Base-2512 (Reason: Ignored word 'base')


### 4. Post-Processing: Prioritize Instruction-Tuned Models
If a base model and its instruction-tuned variant (e.g., `-it`, `-chat`) both exist in the parsed list, we remove the base model and keep only the instruction-tuned version.

In [104]:
model_map = {r["model"]: r for r in rows}
to_remove = {}

for model_id in model_map.keys():
    for s in PRIORITIZE_SUFFIXES:
        s_lower = s.lower()
        model_id_lower = model_id.lower()
        if model_id_lower.endswith(s_lower):
            base_name = model_id[:-len(s)]
            if base_name in model_map:
                to_remove[base_name] = model_id

print("\n🔄 Prioritizing Models:")
if not to_remove:
    print("  - No base models needed removal.")
else:
    for base, tuned in to_remove.items():
        print(f"  ✗ Removed {base}")
        print(f"  ↳ ✓ Kept {tuned}")

final_rows = [r for r in rows if r["model"] not in to_remove]



🔄 Prioritizing Models:
  ✗ Removed Qwen/Qwen1.5-32B
  ↳ ✓ Kept Qwen/Qwen1.5-32B-Chat
  ✗ Removed Qwen/Qwen1.5-MoE-A2.7B
  ↳ ✓ Kept Qwen/Qwen1.5-MoE-A2.7B-Chat
  ✗ Removed Qwen/Qwen1.5-4B
  ↳ ✓ Kept Qwen/Qwen1.5-4B-Chat
  ✗ Removed Qwen/Qwen1.5-0.5B
  ↳ ✓ Kept Qwen/Qwen1.5-0.5B-Chat
  ✗ Removed Qwen/Qwen1.5-14B
  ↳ ✓ Kept Qwen/Qwen1.5-14B-Chat
  ✗ Removed Qwen/Qwen1.5-1.8B
  ↳ ✓ Kept Qwen/Qwen1.5-1.8B-Chat
  ✗ Removed Qwen/Qwen1.5-7B
  ↳ ✓ Kept Qwen/Qwen1.5-7B-Chat
  ✗ Removed Qwen/Qwen2-7B
  ↳ ✓ Kept Qwen/Qwen2-7B-Instruct
  ✗ Removed Qwen/Qwen2-1.5B
  ↳ ✓ Kept Qwen/Qwen2-1.5B-Instruct
  ✗ Removed Qwen/Qwen2-0.5B
  ↳ ✓ Kept Qwen/Qwen2-0.5B-Instruct
  ✗ Removed Qwen/Qwen2.5-0.5B
  ↳ ✓ Kept Qwen/Qwen2.5-0.5B-Instruct
  ✗ Removed Qwen/Qwen2.5-1.5B
  ↳ ✓ Kept Qwen/Qwen2.5-1.5B-Instruct
  ✗ Removed Qwen/Qwen2.5-3B
  ↳ ✓ Kept Qwen/Qwen2.5-3B-Instruct
  ✗ Removed Qwen/Qwen2.5-7B
  ↳ ✓ Kept Qwen/Qwen2.5-7B-Instruct
  ✗ Removed Qwen/Qwen2.5-14B
  ↳ ✓ Kept Qwen/Qwen2.5-14B-Instruct
  ✗ Removed 

### 5. Build DataFrame
We ingest our final collected row definitions into a structured Pandas DataFrame to facilitate easier data processing.

In [105]:
df = pd.DataFrame(final_rows)
df['total_parameters'] = pd.to_numeric(df['total_parameters'], errors='coerce')
df = df.dropna(subset=['total_parameters']).copy()
df.to_csv("models.csv", index=False)

### 6. Calculate Natural Breaks with Jenks
To properly categorize models into distinct size categories without arbitrary thresholds, we use Jenks Natural Breaks optimization. This seeks to minimize each class's average deviation from the class mean, while maximizing each class's deviation from the means of the other groups.

In [106]:
column_data = df['total_parameters']
values = column_data.dropna().astype(float).to_list()
breaks = jenkspy.jenks_breaks(values, n_classes=4)
breaks

[np.float64(0.44),
 np.float64(5.12),
 np.float64(9.65),
 np.float64(15.71),
 np.float64(35.95)]

### 7. Assign Labels
Apply the calculated bins to split our dataframe into 'Tiny/Mobile', 'Small/Edge', 'Standard/Desktop', and 'Mid-range/Server' tiers.

In [107]:
df['category'] = pd.cut(column_data, bins=breaks, labels=["Tiny/Mobile", "Small/Edge", "Standard/Desktop", "Mid-range/Server"], include_lowest=True)
df = df[['collection', 'collection_name', 'model', 'total_parameters', 'activated_parameters', 'max_model_length', 'is_moe', 'is_thinking', 'release_date', 'category']]
df

,collection,collection_name,model,total_parameters,activated_parameters,max_model_length,is_moe,is_thinking,release_date,category
0,Qwen/qwen15,Qwen1.5,Qwen/Qwen1.5-32B-Chat,32.51,32.51,32768,False,False,2024-04-03,Mid-range/Server
1,Qwen/qwen15,Qwen1.5,Qwen/Qwen1.5-MoE-A2.7B-Chat,14.32,2.70,32768,True,False,2024-03-14,Standard/Desktop
2,Qwen/qwen15,Qwen1.5,Qwen/Qwen1.5-4B-Chat,3.95,3.95,32768,False,False,2024-01-30,Tiny/Mobile
3,Qwen/qwen15,Qwen1.5,Qwen/Qwen1.5-0.5B-Chat,0.62,0.62,32768,False,False,2024-01-31,Tiny/Mobile
4,Qwen/qwen15,Qwen1.5,Qwen/Qwen1.5-14B-Chat,14.17,14.17,32768,False,False,2024-01-30,Standard/Desktop
...,...,...,...,...,...,...,...,...,...,...
65,mistralai/ministral-3,Ministral 3,mistralai/Ministral-3-8B-Instruct-2512,8.92,8.92,262144,False,False,2025-10-31,Small/Edge
66,mistralai/ministral-3,Ministral 3,mistralai/Ministral-3-3B-Instruct-2512,3.00,3.00,262144,False,False,2025-10-31,Tiny/Mobile
67,mistralai/ministral-3,Ministral 3,mistralai/Ministral-3-14B-Reasoning-2512,14.00,14.00,262144,False,True,2025-10-31,Standard/Desktop
68,mistralai/ministral-3,Ministral 3,mistralai/Ministral-3-8B-Reasoning-2512,8.00,8.00,262144,False,True,2025-10-31,Small/Edge


### 8. Filter Selection per Family & Size Category

To refine the experimental selection, we apply the following rules within each combination of **size category**, **model family (collection)**, and **architecture type (Dense vs. MoE)**:

1. **Size Priority**: If multiple models exist, select the one with the largest parameter count.
2. **Recency Priority**: If there is a tie in parameter count, select the most recently released model.

The resulting dataset is then saved to a CSV file.

In [108]:
df["total_parameters"] = pd.to_numeric(df["total_parameters"], errors="coerce")
df["activated_parameters"] = pd.to_numeric(df["activated_parameters"], errors="coerce")
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

filtered_df = (
    df.sort_values(
        by=["category", "collection", "total_parameters", "release_date"],
        ascending=[True, True, False, False],
    )
    .drop_duplicates(subset=["category", "collection", "is_moe", "is_thinking"], keep="first")
    .copy()
)

filtered_df.sort_values(by=["category", "release_date"], ascending=[True, True], inplace=True)
filtered_df.reset_index(drop=True, inplace=True)

filtered_df.to_csv('selected_models.csv', index=False)

filtered_df

,collection,collection_name,model,total_parameters,activated_parameters,max_model_length,is_moe,is_thinking,release_date,category
0,Qwen/qwen15,Qwen1.5,Qwen/Qwen1.5-4B-Chat,3.95,3.95,32768,False,False,2024-01-30,Tiny/Mobile
1,google/gemma-release,Gemma release,google/gemma-2b-it,3.03,3.03,8192,False,False,2024-02-08,Tiny/Mobile
2,Qwen/qwen2,Qwen2,Qwen/Qwen2-1.5B-Instruct,1.54,1.54,32768,False,False,2024-06-03,Tiny/Mobile
3,google/gemma-2-release,Gemma 2 Release,google/gemma-2-2b-it,2.61,2.61,8192,False,False,2024-07-16,Tiny/Mobile
4,Qwen/qwen25,Qwen2.5,Qwen/Qwen2.5-3B-Instruct,3.09,3.09,32768,False,False,2024-09-17,Tiny/Mobile
5,meta-llama/llama-32,Llama 3.2,meta-llama/Llama-3.2-3B-Instruct,3.21,3.21,131072,False,False,2024-09-18,Tiny/Mobile
6,deepseek-ai/deepseek-r1,DeepSeek-R1,deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B,1.78,1.78,131072,False,True,2025-01-20,Tiny/Mobile
7,google/gemma-3-release,Gemma 3 Release,google/gemma-3-4b-it,4.30,4.30,131072,False,False,2025-02-20,Tiny/Mobile
8,Qwen/qwen3,Qwen3,Qwen/Qwen3-4B,4.41,4.41,40960,False,True,2025-04-27,Tiny/Mobile
9,google/gemma-3n,Gemma 3n,google/gemma-3n-E4B-it,4.00,4.00,32768,True,False,2025-06-03,Tiny/Mobile


### 9. Final Statistics
Displaying the number of models successfully selected per category.

In [109]:
filtered_df.groupby("category").size().reindex(
    ["Tiny/Mobile", "Small/Edge", "Standard/Desktop", "Mid-range/Server"]
)

category
Tiny/Mobile         15
Small/Edge          15
Standard/Desktop    10
Mid-range/Server    11
dtype: int64

### 10. Select a Representative Model for each Category
These models will be used to approximate the total runtime of the final experiment.

In [110]:
condition = ~filtered_df['is_moe'] & filtered_df['is_thinking']
representative_models = filtered_df[condition].loc[
    filtered_df[condition].groupby('category')['total_parameters'].idxmax()
]
representative_models.to_csv('representative_models.csv', index=False)
representative_models

,collection,collection_name,model,total_parameters,activated_parameters,max_model_length,is_moe,is_thinking,release_date,category
13,Qwen/qwen35,Qwen3.5,Qwen/Qwen3.5-4B,4.66,4.66,262144,False,True,2026-02-27,Tiny/Mobile
28,Qwen/qwen35,Qwen3.5,Qwen/Qwen3.5-9B,9.65,9.65,262144,False,True,2026-02-27,Small/Edge
35,deepseek-ai/deepseek-r1,DeepSeek-R1,deepseek-ai/DeepSeek-R1-Distill-Qwen-14B,14.77,14.77,131072,False,True,2025-01-20,Standard/Desktop
43,deepseek-ai/deepseek-r1,DeepSeek-R1,deepseek-ai/DeepSeek-R1-Distill-Qwen-32B,32.76,32.76,131072,False,True,2025-01-20,Mid-range/Server
